# Map perturbed hormones

In [ ]:
%autosave 0

## Load libaries

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append('/projects/site/pred/ihb-g-deco/USERS/adaml9/sc-ops')
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import delnx as dx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf
from sc_ops.preprocessing._adata_ops import *
from sc_ops.settings._plotting import *

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

# Set the path to the anndata directory
anndata_dir = Path(settings.IO.anndata)

## Setup parameters

In [ ]:
# Define list of hormones
hormones = ["TPH1", "GHRL", "MLN", "GIP", "CCK", "NTS", "GCG", "PYY", "SST"]

# Define a color palette for the EEC subtypes
ct_eec_palette = {
    "EEC Progenitors": "#9467bd",   # purple
    "EC Cells": "#d62728",            # strong red
    "X Cells": "#e377c2",             # magenta
    "D Cells": "#17becf",             # cyan / turquoise
    "K Cells": "#bcbd22",             # olive
    "I/N Cells": "#7f7f7f",           # dark grey
}

# Define on-target cell type of each hormone
hormone_target_celltype = {
    "TPH1": "EC Cells",
    "GHRL": "X Cells",
    "MLN": "X Cells",
    "GIP": "K Cells",
    "GAST": "X Cells",
    "CCK": "I/N Cells",
    "NTS": "I/N Cells",
    "GCG": "I/N Cells",
    "PYY": "I/N Cells",
    "SST": "D Cells",
}

# Generate reverse mapping of cell type to hormones
celltype_hormones = {}
for hormone, celltype in hormone_target_celltype.items():
    if celltype not in celltype_hormones:
        celltype_hormones[celltype] = []
    celltype_hormones[celltype].append(hormone)

In [ ]:
celltype_hormones

## Load data 

In [ ]:
# Load annotated atlas
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered.h5ad")

# Subset to EEC lineage
adata = adata[adata.obs["annotation"].isin(["EEC Progenitors", "EC Cells", "X Cells", "D Cells", "K Cells", "I/N Cells"])]

In [ ]:
# Remove BAMBI from conditions
adata = adata[adata.obs["condition"] != "Bambi"]

In [ ]:
adata.obs["annotation"].unique()

In [ ]:
adata.obs["cystic"].unique()

In [ ]:
adata.X.max()

In [ ]:
# Normalize and log-transform the data
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

### Plot on-target hormone expression for each TF

In [ ]:
# Collect the mean expression of each hormone in the on-target cell type per condition
hormone_mean_expr = []
# For each hormone, we plot the expression of the hormone per condition in the on-target cell type
for hormone in hormone_target_celltype.keys():
    target_celltype = hormone_target_celltype[hormone]
    print(f"Plotting {hormone} in {target_celltype} cells...")
    
    # Subset the data to the target cell type
    adata_subset = adata[adata.obs["annotation"] == target_celltype]

    # Remove conditions with less than 10 cells
    condition_counts = adata_subset.obs["condition"].value_counts()
    valid_conditions = condition_counts[condition_counts >= 10].index
    adata_subset = adata_subset[adata_subset.obs["condition"].isin(valid_conditions)]

    # Sort the conditions based on the mean expression of the hormone in the target cell type
    mean_expr = sc.get.obs_df(adata_subset, keys=hormone, use_raw=False).groupby(adata_subset.obs["condition"]).mean().sort_values(by=hormone, ascending=False)
    
    # Melt the mean expression dataframe for later use
    mean_expr_melt = mean_expr.reset_index().melt(id_vars="condition", value_vars=hormone, var_name="hormone", value_name="mean_expression")
    # Add hormone and target cell type information to the mean expression dataframe
    mean_expr_melt["target_celltype"] = target_celltype

    # Append the mean expression dataframe to the list
    hormone_mean_expr.append(mean_expr_melt)

    # Categories
    categories = list(mean_expr.index)
    # Add all missing conditions to the end of the categories list
    all_conditions = adata.obs["condition"].unique()
    for cond in all_conditions:
        if cond not in categories:
            categories.append(cond)

    # Reorder the conditions in the adata_subset based on the sorted mean expression
    adata_subset.obs["condition"] = pd.Categorical(adata_subset.obs["condition"], categories=categories, ordered=True)
    
    # Define palette by giving each condition the same color
    palette = {cond: ct_eec_palette[target_celltype] for cond in categories}

    # Escape the target cell type name for use in the file name
    target_celltype_escaped = target_celltype.replace("/", "_").replace(" ", "_")

    # Create the violin plot
    with plt.rc_context({"axes.grid": False}):
        fig, ax = plt.subplots(figsize=(12, 2))
        sc.pl.violin(adata_subset, keys=hormone, groupby="condition", palette=palette, use_raw=False, show=True, rotation=90, ax=ax, save=f"_{hormone}_{target_celltype_escaped}.pdf")

### Collect on-target hormone expression and 0-max scale across conditions

In [ ]:
# Collect into a single dataframe
hormone_mean_expr_df = pd.concat(hormone_mean_expr, ignore_index=True)

# Make wider matrix
hormone_mean_expr_wide = hormone_mean_expr_df.pivot_table(index=["hormone", "target_celltype"], columns="condition", values="mean_expression").reset_index()

# Sort based on order of hormones [TPH1, GHRL, MLN, SST, GIP, GAST, CCK, NTS, GCG, PYY]
hormone_order = ["TPH1", "GHRL", "MLN", "SST", "GIP", "GAST", "CCK", "NTS", "GCG", "PYY"]
hormone_mean_expr_wide["hormone"] = pd.Categorical(hormone_mean_expr_wide["hormone"], categories=hormone_order, ordered=True)

# Sort based on hormone order
hormone_mean_expr_wide.sort_values(by=["hormone"], inplace=True)

# Combine hormone and target cell type into a single column for better visualization
hormone_mean_expr_wide["hormone_target"] = hormone_mean_expr_wide["hormone"].astype(str) + " (" + hormone_mean_expr_wide["target_celltype"] + ")"

# Set as index and remove target cell type and hormone columns
hormone_mean_expr_wide.set_index("hormone_target", inplace=True)
hormone_mean_expr_wide.drop(columns=["hormone", "target_celltype"], inplace=True)

# Remove GAST (X Cells) & SST (D Cells)
hormone_mean_expr_wide = hormone_mean_expr_wide[~hormone_mean_expr_wide.index.str.contains("GAST|SST")].fillna(0)

# Scale min max per row 
hormone_mean_expr_wide_scaled = hormone_mean_expr_wide.apply(lambda x: (x) / (x.max()), axis=1)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

cmap = LinearSegmentedColormap.from_list(
    "grey_orrd",
    [
        (0.00, "#d9d9d9"),        # grey for zero
        (0.02, plt.cm.Reds(0.10)),
        (0.20, plt.cm.Reds(0.30)),
        (0.45, plt.cm.Reds(0.50)),
        (0.70, plt.cm.Reds(0.75)),
        (1.00, plt.cm.Reds(0.98)),
    ]
)

norm = Normalize(vmin=0, vmax=1)

h = ma.Heatmap(
    hormone_mean_expr_wide_scaled,
    width=10,
    height=2,
    cmap=cmap,
    norm=norm,
    linewidth=0.3,
)

h.add_legends()
h.add_left(mp.Labels(hormone_mean_expr_wide.index.tolist()))
h.add_bottom(mp.Labels(hormone_mean_expr_wide.columns.tolist()))
h.add_dendrogram("top", method="ward")

with plt.rc_context({"axes.grid": False}):
    h.render()
    plt.savefig(sc.settings.figdir / "hormone_mean_on_target_expression_heatmap.pdf", bbox_inches="tight")


## Plot individual TFs

In [ ]:
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

# Create subfolder inside sc.settings.figdir
output_dir = Path(sc.settings.figdir) / "eec_off_target_hormone_expression"
output_dir.mkdir(parents=True, exist_ok=True)

for tf_cond in adata.obs["condition"].unique():

    adata_subset = adata[
        adata.obs["annotation"].isin(list(ct_eec_palette.keys()))
    ].copy()

    adata_subset = adata_subset[
        adata_subset.obs["condition"].isin(["Control", tf_cond])
    ].copy()

    cells_per_cond = pd.crosstab(
        adata_subset.obs["annotation"],
        adata_subset.obs["condition"]
    )

    valid_cell_types = cells_per_cond[
        (cells_per_cond["Control"] > 10) &
        (cells_per_cond[tf_cond] > 10)
    ].index.tolist()

    pdf_path = (
        output_dir /
        f"eec_hormones_{tf_cond}_vs_control_all_cell_types.pdf"
    )

    with PdfPages(pdf_path) as pdf:

        for cell_type in valid_cell_types:

            adata_celltype = adata_subset[
                adata_subset.obs["annotation"] == cell_type
            ].copy()

            condition_counts = (
                adata_celltype.obs["condition"].value_counts()
            )

            if len(condition_counts) < 2 or condition_counts.min() < 10:
                print(f"Skipping {cell_type}")
                continue

            adata_celltype.obs["condition"] = pd.Categorical(
                adata_celltype.obs["condition"],
                categories=["Control", tf_cond],
                ordered=True
            )

            with plt.rc_context({"axes.grid": False}):

                sc.pl.violin(
                    adata_celltype,
                    keys=hormones,
                    groupby="condition",
                    palette={
                        "Control": "lightgrey",
                        tf_cond: ct_eec_palette[cell_type],
                    },
                    use_raw=False,
                    rotation=90,
                    show=False,
                )

                fig = plt.gcf()

                fig.suptitle(
                    f"{cell_type}: {tf_cond} vs Control",
                    y=1.02
                )

                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

    print(f"Saved {pdf_path}")